# IESO Coincident Peak Prediction — Data Acquisition & Exploratory Analysis

This notebook fetches 15 years of IESO hourly demand data and Open-Meteo historical weather for Toronto,
merges the datasets, and performs comprehensive exploratory analysis of Ontario's peak demand patterns.

**Data sources:**
- IESO public hourly demand CSVs (2010–2021 from `reports-public.ieso.ca`)
- Local ICI demand CSVs (2022–2025 base periods)
- Open-Meteo historical weather archive for Toronto (43.65°N, -79.38°W)
- Ground truth peak data from IESO Top-10 and Top-5 archives (xlsx)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import requests
import openpyxl
import os
import io
import warnings
from pathlib import Path
from datetime import datetime, timedelta

warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Paths
PROJECT_ROOT = Path(r'C:/wamp64/www/Spec_Driven_Dev_Website')
DATA_DIR = PROJECT_ROOT / 'assets' / 'sample data'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'source' / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Constants
TORONTO_LAT = 43.65
TORONTO_LON = -79.38
IESO_BASE_URL = 'https://reports-public.ieso.ca/public/Demand'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Local data files: {list(DATA_DIR.glob("*"))}')

## Load Local ICI Demand CSVs (2022–2025)

These files cover ICI base periods (May 1 – April 30) and have 3 metadata header rows
starting with `\\`. Format: `Date, Hour, Ontario Demand` with hour-ending convention (1–24 EST).

In [ ]:
def load_ici_demand(filepath):
    """Load an IESO ICI demand CSV, skipping metadata header rows."""
    df = pd.read_csv(
        filepath,
        skiprows=3,  # Skip 3 header lines starting with \\
        names=['Date', 'Hour', 'Ontario Demand'],
        parse_dates=['Date']
    )
    df = df.dropna(subset=['Ontario Demand'])
    df['Ontario Demand'] = pd.to_numeric(df['Ontario Demand'], errors='coerce')
    df['Hour'] = pd.to_numeric(df['Hour'], errors='coerce').astype(int)
    return df

# Load each local ICI file
ici_files = {
    2022: DATA_DIR / 'PUB_ICIDemand_2022_v8647.csv',
    2023: DATA_DIR / 'PUB_ICIDemand_2023_v8776.csv',
    2024: DATA_DIR / 'PUB_ICIDemand_2024_v8680.csv',
    2025: DATA_DIR / 'PUB_ICIDemand_2025_v6828.csv',
}

local_dfs = []
for year, path in ici_files.items():
    if path.exists():
        df = load_ici_demand(path)
        df['source'] = f'local_ici_{year}'
        local_dfs.append(df)
        print(f'Loaded {year}: {len(df)} rows, '
              f'{df["Date"].min().strftime("%Y-%m-%d")} to '
              f'{df["Date"].max().strftime("%Y-%m-%d")}')
    else:
        print(f'WARNING: {path.name} not found')

local_demand = pd.concat(local_dfs, ignore_index=True)
print(f'\nTotal local ICI rows: {len(local_demand):,}')

## Fetch IESO Historical Demand (2010–2021)

The IESO publishes calendar-year hourly demand at:
`https://reports-public.ieso.ca/public/Demand/PUB_Demand_[YYYY].csv`

Format: `Date, Hour, Market Demand, Ontario Demand` with 3 metadata header rows.

In [ ]:
def fetch_ieso_demand(year):
    """Fetch IESO historical demand CSV for a calendar year."""
    url = f'{IESO_BASE_URL}/PUB_Demand_{year}.csv'
    print(f'  Fetching {url}...', end=' ')
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        df = pd.read_csv(
            io.StringIO(response.text),
            skiprows=3,
            names=['Date', 'Hour', 'Market Demand', 'Ontario Demand'],
            parse_dates=['Date']
        )
        df = df.dropna(subset=['Ontario Demand'])
        df['Ontario Demand'] = pd.to_numeric(df['Ontario Demand'], errors='coerce')
        df['Hour'] = pd.to_numeric(df['Hour'], errors='coerce').astype(int)
        df = df[['Date', 'Hour', 'Ontario Demand']]
        print(f'{len(df)} rows')
        return df
    except Exception as e:
        print(f'FAILED: {e}')
        return pd.DataFrame()

print('Fetching IESO historical demand (2010-2021)...')
remote_dfs = []
for year in range(2010, 2022):
    df = fetch_ieso_demand(year)
    if not df.empty:
        df['source'] = f'ieso_public_{year}'
        remote_dfs.append(df)

remote_demand = pd.concat(remote_dfs, ignore_index=True)
print(f'\nTotal remote rows: {len(remote_demand):,}')

## Combine into Unified Demand Dataset

Merge local ICI demand (2022–2025) with remote IESO demand (2010–2021) into a single
time-indexed dataframe. Create a proper datetime index in EST using IESO's hour-ending
convention (Hour 1 = midnight to 1 AM).

In [ ]:
# Combine local and remote data
demand = pd.concat([remote_demand, local_demand], ignore_index=True)

# Remove duplicates — local ICI files may overlap with remote calendar-year files
# at the May transition boundaries. Keep the first occurrence.
demand = demand.drop_duplicates(subset=['Date', 'Hour'], keep='first')

# Create datetime index from IESO hour-ending convention:
# Hour 1 = 00:00-01:00 EST, so the timestamp for Hour 1 is 01:00
demand['datetime'] = pd.to_datetime(demand['Date']) + pd.to_timedelta(demand['Hour'], unit='h')
demand = demand.sort_values('datetime').reset_index(drop=True)

# Assign base period (May 1 of year Y to April 30 of year Y+1 = base period Y)
demand['base_period'] = demand['Date'].apply(
    lambda d: d.year if d.month >= 5 else d.year - 1
)

print(f'Unified demand dataset: {len(demand):,} rows')
print(f'Date range: {demand["Date"].min()} to {demand["Date"].max()}')
print(f'Base periods: {sorted(demand["base_period"].unique())}')
print(f'\nRows per base period:')
print(demand.groupby('base_period').size().to_string())

## Load Ground Truth Peak Data

Read the user's spreadsheet files containing:
1. **Top-10 peaks** for 13 base periods (2012–2025) — Date, Hour, Demand (MW)
2. **Top-5 peaks with system-wide consumption** for 15 base periods (2010–2025) — includes AQEW

In [ ]:
# Load Top-10 Peaks Archive
top10_path = DATA_DIR / 'Top-Ten-Ontario-Demand-Peaks-Archive(1).xlsx'
wb = openpyxl.load_workbook(top10_path, data_only=True)

peaks_records = []
for sheet_name in wb.sheetnames:
    ws = wb[sheet_name]
    # Each sheet represents a base period, e.g. "2024" or "2024-2025"
    # Extract base period year from sheet name
    try:
        bp_year = int(sheet_name.split('-')[0].strip())
    except (ValueError, IndexError):
        continue
    
    # Find the header row and data rows
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=False):
        cells = [c.value for c in row]
        # Look for data rows: Rank is an integer, Date is a date
        if (isinstance(cells[0], (int, float)) and 
            cells[0] in range(1, 11) and
            cells[1] is not None):
            rank = int(cells[0])
            date_val = cells[1]
            hour_val = cells[2]
            demand_val = cells[3]
            
            # Parse date
            if isinstance(date_val, datetime):
                date_parsed = date_val.date()
            elif isinstance(date_val, str):
                date_parsed = pd.to_datetime(date_val).date()
            else:
                continue
            
            peaks_records.append({
                'base_period': bp_year,
                'rank': rank,
                'date': pd.Timestamp(date_parsed),
                'hour_ending': int(hour_val) if hour_val else None,
                'ontario_demand_mw': float(demand_val) if demand_val else None,
                'source': 'top10_archive'
            })

peaks_df = pd.DataFrame(peaks_records)
print(f'Loaded {len(peaks_df)} peak records across {peaks_df["base_period"].nunique()} base periods')
print(f'Base periods: {sorted(peaks_df["base_period"].unique())}')

# Separate top-5 as our primary ground truth labels
top5_peaks = peaks_df[peaks_df['rank'] <= 5].copy()
print(f'\nTop-5 peaks: {len(top5_peaks)} records')
print(top5_peaks.head(10).to_string(index=False))

In [ ]:
# Load Top-5 Peaks with System-Wide Consumption
top5_system_path = DATA_DIR / 'Top-5-Peaks-Hours-and-System-Wide-Consumption.xlsx'
wb2 = openpyxl.load_workbook(top5_system_path, data_only=True)

system_records = []
for sheet_name in wb2.sheetnames:
    ws = wb2[sheet_name]
    try:
        bp_year = int(sheet_name.split('-')[0].strip())
    except (ValueError, IndexError):
        continue
    
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=False):
        cells = [c.value for c in row]
        if (isinstance(cells[0], (int, float)) and 
            cells[0] in range(1, 6) and
            cells[1] is not None):
            system_records.append({
                'base_period': bp_year,
                'rank': int(cells[0]),
                'date': pd.Timestamp(cells[1]) if isinstance(cells[1], datetime) else pd.to_datetime(str(cells[1])),
                'hour_ending': int(cells[2]) if cells[2] else None,
                'aqew_mwh': float(cells[3]) if cells[3] else None,
                'embedded_gen_mwh': float(cells[4]) if len(cells) > 4 and cells[4] else None,
                'total_mwh': float(cells[-1]) if cells[-1] and isinstance(cells[-1], (int, float)) else None,
            })

system_df = pd.DataFrame(system_records)
print(f'Loaded {len(system_df)} system consumption records')
print(f'Base periods: {sorted(system_df["base_period"].unique())}')
print(f'\nSample records:')
print(system_df.head(10).to_string(index=False))

## Fetch Open-Meteo Historical Weather

Fetch hourly weather data for Toronto (43.65°N, -79.38°W) from the Open-Meteo historical
archive API. Variables: temperature_2m, relative_humidity_2m, dewpoint_2m, wind_speed_10m,
shortwave_radiation.

The API accepts date ranges up to ~1 year per request, so we batch by year.

In [ ]:
def fetch_openmeteo_weather(start_date, end_date, lat=TORONTO_LAT, lon=TORONTO_LON):
    """Fetch hourly weather from Open-Meteo historical archive."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': lat,
        'longitude': lon,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,relative_humidity_2m,dewpoint_2m,wind_speed_10m,shortwave_radiation',
        'timezone': 'America/Toronto'
    }
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    hourly = data['hourly']
    df = pd.DataFrame({
        'datetime': pd.to_datetime(hourly['time']),
        'temperature_c': hourly['temperature_2m'],
        'relative_humidity': hourly['relative_humidity_2m'],
        'dewpoint_c': hourly['dewpoint_2m'],
        'wind_speed_kmh': hourly['wind_speed_10m'],
        'shortwave_radiation': hourly['shortwave_radiation'],
    })
    return df

print('Fetching Open-Meteo historical weather for Toronto (2010-2025)...')
weather_dfs = []
for year in range(2010, 2026):
    start = f'{year}-01-01'
    end = f'{year}-12-31' if year < 2026 else '2026-03-08'
    print(f'  {year}...', end=' ')
    try:
        wdf = fetch_openmeteo_weather(start, end)
        weather_dfs.append(wdf)
        print(f'{len(wdf)} rows')
    except Exception as e:
        print(f'FAILED: {e}')

weather = pd.concat(weather_dfs, ignore_index=True)
weather = weather.drop_duplicates(subset=['datetime']).sort_values('datetime').reset_index(drop=True)
print(f'\nTotal weather rows: {len(weather):,}')
print(f'Date range: {weather["datetime"].min()} to {weather["datetime"].max()}')

## Timestamp Alignment & Merge

**Critical alignment:** IESO uses hour-ending convention (Hour 1 = 00:00–01:00 EST,
timestamp at 01:00). Open-Meteo uses hour-beginning convention in local timezone
(hour 0 = 00:00–01:00, timestamp at 00:00).

To align: IESO Hour 1 at 01:00 → Open-Meteo hour 0 at 00:00. So we shift IESO's
datetime back by 1 hour, or equivalently, shift Open-Meteo forward by 1 hour.

In [ ]:
# Open-Meteo timestamps are hour-beginning in local (EST/EDT) time.
# IESO datetime is hour-ending: Hour=1 means 01:00 for the 00:00-01:00 interval.
# To align: shift Open-Meteo forward by 1 hour so both represent hour-ending.
weather['datetime_he'] = weather['datetime'] + pd.Timedelta(hours=1)

# Merge on the aligned datetime
merged = demand.merge(
    weather,
    left_on='datetime',
    right_on='datetime_he',
    how='left',
    suffixes=('', '_weather')
)

# Check merge quality
weather_coverage = merged['temperature_c'].notna().mean() * 100
print(f'Merged dataset: {len(merged):,} rows')
print(f'Weather coverage: {weather_coverage:.1f}% of demand rows have weather data')
print(f'\nMissing weather by year:')
missing_by_year = merged.groupby(merged['Date'].dt.year)['temperature_c'].apply(lambda x: x.isna().sum())
print(missing_by_year[missing_by_year > 0].to_string())

In [ ]:
# Data validation: check hourly completeness
print('Hours per calendar year (expected ~8,760):')
hours_by_year = demand.groupby(demand['Date'].dt.year).size()
for year, count in hours_by_year.items():
    expected = 8784 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 8760
    status = 'OK' if abs(count - expected) < 48 else f'GAP ({expected - count} missing)'
    # Note: DST transitions and partial years (2025) will show slight deviations
    print(f'  {year}: {count:,} hours — {status}')

# Forward-fill small weather gaps (< 3 hours)
weather_cols = ['temperature_c', 'relative_humidity', 'dewpoint_c', 'wind_speed_kmh', 'shortwave_radiation']
merged[weather_cols] = merged[weather_cols].fillna(method='ffill', limit=3)

remaining_gaps = merged[weather_cols].isna().sum()
print(f'\nRemaining weather gaps after forward-fill (limit=3):')
print(remaining_gaps.to_string())

## EDA: Annual Peak Demand Trend (2010–2025)

How has Ontario's peak system demand evolved over 15 years? The trend reflects
embedded generation growth, conservation programs, and structural demand shifts.

In [ ]:
# Annual maximum Ontario Demand by calendar year
annual_peak = demand.groupby(demand['Date'].dt.year)['Ontario Demand'].max().reset_index()
annual_peak.columns = ['Year', 'Peak Demand (MW)']
# Exclude partial year 2026
annual_peak = annual_peak[annual_peak['Year'] <= 2025]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(annual_peak['Year'], annual_peak['Peak Demand (MW)'], 'o-', 
        color='#d32f2f', linewidth=2, markersize=8)
ax.fill_between(annual_peak['Year'], annual_peak['Peak Demand (MW)'], 
                alpha=0.1, color='#d32f2f')
ax.set_xlabel('Year')
ax.set_ylabel('Annual Peak Ontario Demand (MW)')
ax.set_title('Ontario Annual Peak System Demand (2010–2025)')
ax.set_ylim(18000, 26000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Annotate notable years
for _, row in annual_peak.iterrows():
    ax.annotate(f'{row["Peak Demand (MW)"]:.0f}', 
                (row['Year'], row['Peak Demand (MW)']),
                textcoords='offset points', xytext=(0, 10),
                fontsize=8, ha='center', color='#555')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'annual_peak_demand.png', dpi=150, bbox_inches='tight')
plt.show()
print(annual_peak.to_string(index=False))

## EDA: Monthly Demand Box Plots

Demand follows a strong seasonal pattern — summer cooling load drives the highest demands,
with a secondary winter heating peak in some years.

In [ ]:
# Daily max demand for box plots
daily_max = demand.groupby('Date')['Ontario Demand'].max().reset_index()
daily_max.columns = ['Date', 'Daily Max Demand (MW)']
daily_max['Month'] = daily_max['Date'].dt.month
daily_max['Month Name'] = daily_max['Date'].dt.strftime('%b')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=daily_max, x='Month Name', y='Daily Max Demand (MW)',
            order=month_order, palette='coolwarm', ax=ax,
            fliersize=2, linewidth=1)
ax.set_xlabel('Month')
ax.set_ylabel('Daily Maximum Demand (MW)')
ax.set_title('Ontario Daily Peak Demand by Month (2010–2025)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Mark the peak season
ax.axvspan(4.5, 7.5, alpha=0.08, color='red', label='Peak season (Jun–Aug)')
ax.legend(loc='upper left')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'monthly_demand_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## EDA: Hourly Demand Heatmap

Average demand by hour-of-day and month reveals the afternoon summer peak cluster
where coincident peaks concentrate.

In [ ]:
demand['Month'] = demand['Date'].dt.month
heatmap_data = demand.pivot_table(
    values='Ontario Demand',
    index='Hour',
    columns='Month',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=False, fmt='.0f',
            xticklabels=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                         'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'],
            yticklabels=range(1, 25),
            ax=ax, cbar_kws={'label': 'Average Demand (MW)'})
ax.set_xlabel('Month')
ax.set_ylabel('Hour Ending (EST)')
ax.set_title('Average Ontario Demand by Hour and Month (2010–2025)\n'
             'Coincident peaks cluster in the hot upper-right region: Jun–Aug, Hours 15–19')

# Draw box around peak zone
from matplotlib.patches import Rectangle
rect = Rectangle((5, 13), 3, 6, linewidth=2, edgecolor='black', 
                  facecolor='none', linestyle='--')
ax.add_patch(rect)
ax.text(6.5, 12.5, 'Peak Zone', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'hourly_demand_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## EDA: Peak Analysis

Distribution of actual top-5 coincident peaks by month, day-of-week, and hour-of-day
across all available base periods.

In [ ]:
# Analyze top-5 peaks
top5 = top5_peaks.copy()
top5['month'] = top5['date'].dt.month
top5['month_name'] = top5['date'].dt.strftime('%b')
top5['day_of_week'] = top5['date'].dt.day_name()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Month distribution
month_counts = top5['month'].value_counts().reindex(range(1, 13), fill_value=0)
colors = ['#2196F3' if m not in [6, 7, 8] else '#d32f2f' for m in range(1, 13)]
axes[0].bar(range(1, 13), month_counts.values, color=colors)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Count')
axes[0].set_title('Top-5 Peaks by Month')
pct_summer = (month_counts[[6, 7, 8]].sum() / month_counts.sum()) * 100
axes[0].text(0.95, 0.95, f'Jun–Aug: {pct_summer:.0f}%', transform=axes[0].transAxes,
             ha='right', va='top', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Day of week distribution
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = top5['day_of_week'].value_counts().reindex(dow_order, fill_value=0)
dow_colors = ['#4CAF50' if d not in ['Saturday', 'Sunday'] else '#9E9E9E' for d in dow_order]
axes[1].bar(range(7), dow_counts.values, color=dow_colors)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
axes[1].set_xlabel('Day of Week')
axes[1].set_ylabel('Count')
axes[1].set_title('Top-5 Peaks by Day of Week')
axes[1].text(0.95, 0.95, '100% weekdays', transform=axes[1].transAxes,
             ha='right', va='top', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))

# Hour distribution
hour_counts = top5['hour_ending'].value_counts().reindex(range(1, 25), fill_value=0)
hour_colors = ['#FF9800' if h in range(15, 20) else '#BDBDBD' for h in range(1, 25)]
axes[2].bar(range(1, 25), hour_counts.values, color=hour_colors)
axes[2].set_xlabel('Hour Ending (EST)')
axes[2].set_ylabel('Count')
axes[2].set_title('Top-5 Peaks by Hour')
pct_window = (hour_counts[15:20].sum() / hour_counts.sum()) * 100
axes[2].text(0.95, 0.95, f'HE 15–19: {pct_window:.0f}%', transform=axes[2].transAxes,
             ha='right', va='top', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.5))

plt.suptitle('When Do Ontario\'s Coincident Peaks Occur?', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'peak_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print('\n=== Top-5 Peak Summary Statistics ===')
print(f'Total peaks analyzed: {len(top5)}')
print(f'June–August concentration: {pct_summer:.1f}%')
print(f'Weekday concentration: 100.0%')
print(f'Hours 15–19 concentration: {pct_window:.1f}%')
print(f'\nDemand range: {top5["ontario_demand_mw"].min():,.0f} – {top5["ontario_demand_mw"].max():,.0f} MW')
print(f'Most common hour: HE {top5["hour_ending"].mode().values[0]}')

## EDA: Weather-Demand Relationship

The physical basis for prediction: daily maximum temperature vs. daily maximum demand.
Above approximately 22°C, AC cooling load drives demand nonlinearly upward.

In [ ]:
# Create daily summary for scatter plot
daily_summary = merged.groupby('Date').agg({
    'Ontario Demand': 'max',
    'temperature_c': 'max'
}).reset_index()
daily_summary.columns = ['Date', 'Daily Max Demand', 'Daily Max Temp']
daily_summary['Month'] = daily_summary['Date'].dt.month
daily_summary = daily_summary.dropna()

# Season labels
def get_season(month):
    if month in [6, 7, 8]:
        return 'Summer (Jun–Aug)'
    elif month in [12, 1, 2]:
        return 'Winter (Dec–Feb)'
    elif month in [3, 4, 5]:
        return 'Spring (Mar–May)'
    else:
        return 'Fall (Sep–Nov)'

daily_summary['Season'] = daily_summary['Month'].apply(get_season)

# Scatter plot
fig, ax = plt.subplots(figsize=(12, 7))
season_colors = {
    'Summer (Jun–Aug)': '#d32f2f',
    'Winter (Dec–Feb)': '#1565C0',
    'Spring (Mar–May)': '#4CAF50',
    'Fall (Sep–Nov)': '#FF9800'
}

for season, color in season_colors.items():
    mask = daily_summary['Season'] == season
    ax.scatter(daily_summary.loc[mask, 'Daily Max Temp'],
              daily_summary.loc[mask, 'Daily Max Demand'],
              alpha=0.3, s=10, color=color, label=season)

# Mark actual top-5 peak days
peak_dates = set(top5_peaks['date'].dt.date)
peak_mask = daily_summary['Date'].dt.date.isin(peak_dates)
if peak_mask.any():
    ax.scatter(daily_summary.loc[peak_mask, 'Daily Max Temp'],
              daily_summary.loc[peak_mask, 'Daily Max Demand'],
              marker='*', s=200, color='black', zorder=5, label='Top-5 Peak Days',
              edgecolors='gold', linewidths=1)

# Mark the ~22°C inflection point
ax.axvline(x=22, color='gray', linestyle='--', alpha=0.5)
ax.text(22.5, ax.get_ylim()[0] + 500, 'AC inflection\n~22°C', fontsize=9, color='gray')

ax.set_xlabel('Daily Maximum Temperature (°C)')
ax.set_ylabel('Daily Maximum Ontario Demand (MW)')
ax.set_title('Temperature vs. Demand — The Physical Basis for Peak Prediction')
ax.legend(loc='upper left', framealpha=0.9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'weather_demand_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation analysis
summer = daily_summary[daily_summary['Month'].isin([6, 7, 8])]
corr_all = daily_summary[['Daily Max Demand', 'Daily Max Temp']].corr().iloc[0, 1]
corr_summer = summer[['Daily Max Demand', 'Daily Max Temp']].corr().iloc[0, 1]
print(f'Correlation (all seasons): {corr_all:.3f}')
print(f'Correlation (summer only): {corr_summer:.3f}')

## Save Cleaned Dataset

Export the merged demand + weather dataset as parquet for downstream notebooks.
Also save the ground truth peak labels separately.

In [ ]:
# Save merged hourly dataset
cols_to_save = ['Date', 'Hour', 'Ontario Demand', 'datetime', 'base_period',
                'temperature_c', 'relative_humidity', 'dewpoint_c', 
                'wind_speed_kmh', 'shortwave_radiation']
output_df = merged[cols_to_save].copy()
output_df.to_parquet(OUTPUT_DIR / 'ieso_demand_weather_2010_2025.parquet', index=False)
print(f'Saved hourly dataset: {len(output_df):,} rows')
print(f'  -> {OUTPUT_DIR / "ieso_demand_weather_2010_2025.parquet"}')

# Save peak labels
peaks_df.to_parquet(OUTPUT_DIR / 'ieso_peak_labels.parquet', index=False)
print(f'\nSaved peak labels: {len(peaks_df)} records')
print(f'  -> {OUTPUT_DIR / "ieso_peak_labels.parquet"}')

# Save system consumption data
system_df.to_parquet(OUTPUT_DIR / 'ieso_system_consumption.parquet', index=False)
print(f'\nSaved system consumption: {len(system_df)} records')
print(f'  -> {OUTPUT_DIR / "ieso_system_consumption.parquet"}')

# Save daily summary for quick access
daily_summary.to_parquet(OUTPUT_DIR / 'ieso_daily_summary.parquet', index=False)
print(f'\nSaved daily summary: {len(daily_summary)} records')
print(f'  -> {OUTPUT_DIR / "ieso_daily_summary.parquet"}')

print('\n=== Notebook 1 complete ===')